# AMD ROCm Hipify Tool Demo

This notebook demostrate how to translate `depthwise_conv3d.cu` with **hipify-perl**, compile with **hipcc**, and run the benchmark on an AMD GPU.

**Pipeline:** `.cu` -> hipify-perl -> `.hip` -> hipcc -> `depthwise_conv3d` executable.

## Prerequisites

| Step | Requirement |
|------|-------------|
| Hipify + HIP build | ROCm installed; **`hipify`** and **`hipcc`**|

**Tip:** Run the first code cell from the repo root or from `hipify/` so it can locate this demo directory.

## Demo layout

```text
hipify/
  depthwise_conv3d.cu   # CUDA source
  run_hipify.sh         # CUDA -> HIP
  build_kernel.sh       # hipcc build
  run_kernel.sh         # run built hip kernel
  hipify_demo.ipynb     # this notebook
```

## What the CUDA program does

`depthwise_conv3d.cu` is a **depthwise 3D convolution** benchmark in **BF16**: device allocation, a **portable reference kernel** (grid-stride), GPU event timing, and optional **CPU float32** correctness check.

In [1]:
import os
import subprocess
from pathlib import Path


def find_hipify_demo_dir() -> Path:
    # Folder that contains run_hipify.sh and depthwise_conv3d.cu
    here = Path.cwd().resolve()
    for p in (here, here / "hipify"):
        if (p / "run_hipify.sh").is_file() and (p / "depthwise_conv3d.cu").is_file():
            return p
    for parent in (here, *here.parents):
        p = parent / "hipify"
        if (p / "run_hipify.sh").is_file() and (p / "depthwise_conv3d.cu").is_file():
            return p
    raise FileNotFoundError(
        "Could not find hipify demo directory (need run_hipify.sh + depthwise_conv3d.cu). "
        "cd into hipify/ or the repo root and re-run this cell."
    )


HIPIFY_DIR = find_hipify_demo_dir()
os.chdir(HIPIFY_DIR)
print("Working directory:", HIPIFY_DIR)

Working directory: /root/workspace/bytedance/demo/force_demo/hipify_demo


## Step 1 — Translate CUDA to HIP (`run_hipify.sh`)

Runs **hipify-perl** on `depthwise_conv3d.cu`, writes `depthwise_conv3d.hip`, and tees statistics to **`hipify.log`**. The next cell **only prints** `run_hipify.sh` for reference; the cell after that **executes** it.

In [2]:
hipify_script = HIPIFY_DIR / "run_hipify.sh"
hipify_script.chmod(hipify_script.stat().st_mode | 0o111)
subprocess.run(["bash", str(hipify_script)], cwd=HIPIFY_DIR, check=True)
print("hipify-perl finished.")

perl: warning: Setting locale failed.
perl: warning: Please check that your locale settings:
	LANGUAGE = (unset),
	LC_ALL = (unset),
	LC_CTYPE = "C.UTF-8",
	LANG = "en_US.UTF-8"
    are supported and installed on your system.
perl: warning: Falling back to the standard locale ("C").

[HIPIFY] info: file 'depthwise_conv3d.cu' statistics:
  CONVERTED refs count: 80
  TOTAL lines of code: 436
  WARNINGS: 0
[HIPIFY] info: CONVERTED refs by names:
  __nv_bfloat16 => __hip_bfloat16: 35
  cudaDeviceProp => hipDeviceProp_t: 1
  cudaDeviceSynchronize => hipDeviceSynchronize: 1
  cudaError_t => hipError_t: 1
  cudaEventCreate => hipEventCreate: 2
  cudaEventDestroy => hipEventDestroy: 4
  cudaEventElapsedTime => hipEventElapsedTime: 1
  cudaEventRecord => hipEventRecord: 2
  cudaEventSynchronize => hipEventSynchronize: 1
  cudaEvent_t => hipEvent_t: 1
  cudaFree => hipFree: 8
  cudaGetDevice => hipGetDevice: 1
  cudaGetDeviceProperties => hipGetDeviceProperties: 1
  cudaGetErrorString => hipGetE

### Compare `.cu` vs `.hip` around `main`

The next cell prints **lines 368–380** (1-based, inclusive) from both `depthwise_conv3d.cu` and `depthwise_conv3d.hip` so you can see how hipify rewrote the host driver (e.g. `cuda*` → `hip*`). Change **`MAIN_LO`** / **`MAIN_HI`** in that cell if `main` moves after you edit the sources.

In [3]:
# Compare `main` (1-based line numbers, inclusive).
MAIN_LO, MAIN_HI = 368, 380


def lines_range(path: Path, lo: int, hi: int) -> str:
    """Lines lo..hi (1-based inclusive). Clamps to file length."""
    lines = path.read_text().splitlines()
    n = len(lines)
    a = max(0, lo - 1)
    b = min(n, hi)
    body = "\n".join(f"{i + 1:4d} | {lines[i]}" for i in range(a, b))
    return f"=== {path.name} lines {max(lo, 1)}-{min(hi, n)} (of {n}) ===\n{body}"


cu = HIPIFY_DIR / "depthwise_conv3d.cu"
hip = HIPIFY_DIR / "depthwise_conv3d.hip"
print(lines_range(cu, MAIN_LO, MAIN_HI))
print()
print(lines_range(hip, MAIN_LO, MAIN_HI))

=== depthwise_conv3d.cu lines 368-380 (of 436) ===
 368 |   __nv_bfloat16 *d_in = nullptr, *d_out = nullptr, *d_w = nullptr, *d_bias = nullptr;
 369 |   CUDA_CHECK(cudaMalloc(&d_in, n_in * sizeof(__nv_bfloat16)));
 370 |   CUDA_CHECK(cudaMalloc(&d_out, n_out * sizeof(__nv_bfloat16)));
 371 |   CUDA_CHECK(cudaMalloc(&d_w, n_w * sizeof(__nv_bfloat16)));
 372 |   CUDA_CHECK(cudaMalloc(&d_bias, n_bias * sizeof(__nv_bfloat16)));
 373 | 
 374 |   CUDA_CHECK(cudaMemcpy(d_in, h_in.data(), n_in * sizeof(__nv_bfloat16), cudaMemcpyHostToDevice));
 375 |   CUDA_CHECK(cudaMemcpy(d_w, h_w.data(), n_w * sizeof(__nv_bfloat16), cudaMemcpyHostToDevice));
 376 |   CUDA_CHECK(cudaMemcpy(d_bias, h_bias.data(), n_bias * sizeof(__nv_bfloat16), cudaMemcpyHostToDevice));
 377 |   CUDA_CHECK(cudaMemset(d_out, 0, n_out * sizeof(__nv_bfloat16)));
 378 | 
 379 |   int dev_id = 0;
 380 |   CUDA_CHECK(cudaGetDevice(&dev_id));

=== depthwise_conv3d.hip lines 368-380 (of 437) ===
 368 | 
 369 |   __hip_bfloat16 *d_in 

## Step 2 — Compile HIP (`build_kernel.sh`)

Uses **hipcc** with **`-std=c++17`** and **`--offload-arch`**. Set **`HIP_ARCH`** (e.g. `gfx942`) before running the next cell if autodetect is wrong (e.g. `os.environ["HIP_ARCH"] = "gfx942"` in a cell above, or export in the shell that launched Jupyter).

In [4]:
build = HIPIFY_DIR / "build_kernel.sh"
build.chmod(build.stat().st_mode | 0o111)
env = {**os.environ}
# os.environ.setdefault("HIP_ARCH", "gfx942")  # uncomment to force an arch
subprocess.run(["bash", str(build)], cwd=HIPIFY_DIR, env=env, check=True)
exe = HIPIFY_DIR / "depthwise_conv3d"
assert exe.is_file(), "depthwise_conv3d binary missing"
print("Binary:", exe)

Built: /root/workspace/bytedance/demo/force_demo/hipify_demo/depthwise_conv3d (HIP_ARCH=gfx942)
Binary: /root/workspace/bytedance/demo/force_demo/hipify_demo/depthwise_conv3d


## Step 3 — Run the HIP binary

Arguments: **`[niter] [nwarmup] [--no-check]`**  
Below: **`2`** timed iterations, **`2`** warmup (quick demo). CLI defaults are `10` / `10`.

In [5]:
exe = HIPIFY_DIR / "depthwise_conv3d"
proc = subprocess.run(
    [str(exe), "2", "2"],
    cwd=HIPIFY_DIR,
    capture_output=True,
    text=True,
)
print(proc.stdout)
if proc.stderr:
    print("--- stderr ---")
    print(proc.stderr)
proc.check_returncode()


=== Depthwise Conv3D BF16 (portable reference kernel; hipify CUDA->HIP demo) ===
Device id 0: 
  multiProcessorCount=80  maxThreadsPerBlock=1024  warpSize=64

--- Problem (case3-style depthwise) ---
  Input NCHW:   [1, 512, 61, 45, 80]  BF16
  Weight:       [512, 1, 3, 5, 5]  BF16  (groups=512 depthwise)
  Output NCHW:  [1, 512, 59, 45, 80]  BF16
  stride=(1,1,1)  dilation=(1,1,1)  padding=(0,2,2)

--- Device buffers ---
  input           224870400 B  (  214.45 MiB)
  output          217497600 B  (  207.42 MiB)
  weights             76800 B  (    0.07 MiB)
  bias                 1024 B  (    0.00 MiB)
  total alloc     442445824 B  (  421.95 MiB)

--- Kernel configuration ---
  Kernel: conv_depthwise3d_cuda_kernel_reference_bf16 (portable reference)
  Compile-time: KD=3 KH=5 KW=5  PaddingD/H/W=0/2/2  BLOCK_H=45 BLOCK_W=80  BF16
  Block:  (256, 1, 1) -> 256 threads / block
  Grid:   (424800, 1, 1) -> 424800 blocks  (num_output=108748800, ~108748800 thread slots / launch)

Running warmu

## Troubleshooting

| Symptom | What to try |
|---------|-------------|
| `depthwise_conv3d.hip` missing | Re-run Step 1 (`run_hipify.sh`) from the `hipify/` directory. |
| `hipify-perl` not found | Edit `run_hipify.sh` if ROCm is not under `/opt/rocm`. |
| hipcc / ISA errors | Set **`HIP_ARCH`** to your GPU (see `/opt/rocm/bin/rocminfo` for `gfx*`). |
| Empty device name in printed header | Often cosmetic on HIP; check timing and correctness line. |